# EchoVision -- Perception & Context Module

**คำอธิบายภาษาไทย:** โมเดล Multi-task ที่เทรนจากศูนย์ (ไม่มี pretrained weight, ไม่ใช้ข้อมูลภายนอก) รับ input เป็นภาพจากกล้องแว่นตา + เสียงสะท้อน ultrasonic แล้วทำนาย 4 อย่างพร้อมกัน: ประเภทสิ่งกีดขวาง (Object_Type), ระยะห่าง (distance), บริเวณสถานที่ (Location_Zone), และสภาพแสง (Illumination)

---

---

**English original below:**

# EchoVision -- Perception & Context Module

Multi-task model trained **from scratch** (random initialisation only, no pretrained
weights, no external data) that predicts, from a wearable camera image + a 40 kHz
ultrasonic echo recording:

- `Object_Type` (5-way classification)
- `distance` (regression, metres)
- `Location_Zone` (4-way classification)
- `Illumination` (3-way classification)

Perception Score = 0.4*MacroF1(Object_Type) + 0.3*(1 - RMSE(distance)/2) +
0.2*MacroF1(Location_Zone) + 0.1*MacroF1(Illumination)


In [ ]:
import os, sys, time, argparse
import numpy as np
import pandas as pd
import cv2
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from concurrent.futures import ProcessPoolExecutor

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DATASET = "dataset"          # extracted competition zip: images/, audio/, train.csv, val.csv, test.csv
PROCESSED = "dataset_processed"
WEIGHTS_DIR = "outputs/weights"
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

IMG_SIZE = 160
OBJECT_TYPES = ["Large Obstacle", "Small/Low Obstacle", "Elevation Change", "Structural", "Clear Path"]
LOCATION_ZONES = ["Living Room", "Home Office", "Kitchen", "Corridor"]
ILLUMINATIONS = ["Bright", "Dim", "Dark"]
OBJ2I = {v: i for i, v in enumerate(OBJECT_TYPES)}
ZONE2I = {v: i for i, v in enumerate(LOCATION_ZONES)}
ILLUM2I = {v: i for i, v in enumerate(ILLUMINATIONS)}
TASK_WEIGHTS = {"object_type": 0.4, "distance": 0.3, "location_zone": 0.2, "illumination": 0.1}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. โครงสร้างของสัญญาณเสียง (Audio structure)

เซนเซอร์ ultrasonic สุ่มสัญญาณที่ 100 kHz แต่ละไฟล์เสียงคือชุดคาบ (ping) ยาว 3000 samples (30 ms) ที่ยิงซ้ำๆ 378 หรือ 504 ครั้ง — 120 samples แรกคือสัญญาณที่ยิงออกไป (transmit burst) ส่วนที่เหลือคือช่วงรอฟัง echo สะท้อนกลับ

เวลาที่ echo กลับมาบอกระยะทาง (time-of-flight) ดังนั้นแทนที่จะแปลงเป็น mel-spectrogram (ซึ่งจะเฉลี่ยทำลาย timing ที่ละเอียดทิ้ง) เราแปลงแต่ละไฟล์เป็น **B-scan** แทน คือภาพ 2 มิติของ (ลำดับ ping x ช่วงระยะ) พร้อมกราฟ onset ของแต่ละ ping แยกต่างหาก

---

---

**English original below:**

## 1. Audio structure

The ultrasonic sensor samples at 100 kHz. Each recording is a sequence of 3000-sample
(30 ms) ping periods: the first ~120 samples are the 40 kHz transmit burst, the rest
is the listen window where an echo may return. Echo arrival time encodes range, so
instead of a mel-spectrogram (which would average away that timing) each file is
turned into a **B-scan**: a 2D image of (ping index x range bin), plus an explicit
first-echo-onset curve per ping.


In [ ]:
PING = 3000
SKIP = 120
RANGE_BINS = 144
BIN_W = 20
N_PINGS = 256
USED = RANGE_BINS * BIN_W

def build_bscan(path):
    try:
        y, sr = sf.read(path, dtype="float32")
    except Exception:
        return (np.zeros((N_PINGS, RANGE_BINS), np.float16), np.zeros(N_PINGS, np.float16))
    if y.ndim > 1:
        y = y.mean(1)
    n = len(y) // PING
    if n == 0:
        return (np.zeros((N_PINGS, RANGE_BINS), np.float16), np.zeros(N_PINGS, np.float16))
    W = y[: n * PING].reshape(n, PING)
    tx = np.abs(W[:, :SKIP]).max(1, keepdims=True) + 1e-6
    listen = np.abs(W[:, SKIP:SKIP + USED])
    b = listen.reshape(n, RANGE_BINS, BIN_W)
    E = np.sqrt((b.astype(np.float32) ** 2).mean(2))
    E = E / tx
    floor = np.median(E, axis=1, keepdims=True)
    above = E > floor * 6.0
    onset = np.where(above.any(1), above.argmax(1), RANGE_BINS).astype(np.float32) / RANGE_BINS
    E = np.log1p(E * 500.0)
    idx = np.linspace(0, n - 1, N_PINGS).round().astype(np.int64)
    return E[idx].astype(np.float16), onset[idx].astype(np.float16)

def process_image(path, size=IMG_SIZE):
    # cv2.imread uses the ANSI codepage for the path on Windows, which corrupts
    # non-ASCII directory names and silently fails the read. Decode through
    # Python file I/O instead so this works regardless of the OS/locale.
    with open(path, "rb") as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if img is None:
        return np.zeros((size, size, 3), dtype=np.uint8)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


## 2. Preprocess ข้อมูลแต่ละ split ครั้งเดียว เก็บเป็นไฟล์ .npy ขนาดเล็ก

ทำแบบนี้เพื่อไม่ต้อง decode ไฟล์ PNG/FLAC ใหม่ทุก epoch (เร็วกว่ามาก)

---

---

**English original below:**

## 2. Pre-process each split once into compact .npy arrays (avoids re-decoding PNG/FLAC every epoch)

In [ ]:
def _worker(args):
    i, img_path, aud_path = args
    bs, on = build_bscan(aud_path)
    return i, process_image(img_path), bs, on

def build_split(split, workers=8):
    df = pd.read_csv(os.path.join(DATASET, f"{split}.csv"))
    n = len(df)
    img_dir = os.path.join(DATASET, "images")
    aud_dir = os.path.join(DATASET, "audio")
    tasks = [(i, os.path.join(img_dir, df["image name"].iloc[i]), os.path.join(aud_dir, df["audio name"].iloc[i])) for i in range(n)]

    images = np.zeros((n, IMG_SIZE, IMG_SIZE, 3), np.uint8)
    bscans = np.zeros((n, N_PINGS, RANGE_BINS), np.float16)
    onsets = np.zeros((n, N_PINGS), np.float16)

    done = 0
    with ProcessPoolExecutor(max_workers=workers) as ex:
        for i, im, bs, on in ex.map(_worker, tasks, chunksize=8):
            images[i], bscans[i], onsets[i] = im, bs, on
            done += 1
            if done % 1000 == 0:
                print(f"  {split}: {done}/{n}")

    np.save(os.path.join(PROCESSED, f"{split}_images.npy"), images)
    np.save(os.path.join(PROCESSED, f"{split}_bscan.npy"), bscans)
    np.save(os.path.join(PROCESSED, f"{split}_onset.npy"), onsets)
    print(f"{split}: images{images.shape} bscan{bscans.shape} onset{onsets.shape}")
    return df

# Run once. Skips automatically if the .npy files already exist.
for split in ["train", "val", "test"]:
    if not os.path.exists(os.path.join(PROCESSED, f"{split}_images.npy")):
        build_split(split)
    else:
        print(f"{split}: already processed")


## 3. Dataset พร้อม Augmentation (จำลองสภาพเซนเซอร์ที่ไม่สมบูรณ์)

จำลองสถานการณ์จริงตามโจทย์: ภาพมืด/สว่างเกินไป (brightness/gamma jitter), สัญญาณเสียงมี noise หรือขาดหายบางช่วง (ping-axis masking = จำลอง sensor dropout)

---

---

**English original below:**

## 3. Dataset with augmentation (brightness/gamma/noise for images, ping-axis masking + dropout for the sonar)

In [ ]:
def load_split(split):
    df = pd.read_csv(os.path.join(DATASET, f"{split}.csv"))
    images = np.load(os.path.join(PROCESSED, f"{split}_images.npy"), mmap_mode="r")
    bscan = np.load(os.path.join(PROCESSED, f"{split}_bscan.npy"), mmap_mode="r")
    onset = np.load(os.path.join(PROCESSED, f"{split}_onset.npy"), mmap_mode="r")
    return df, images, bscan, onset

class EchoDataset(Dataset):
    def __init__(self, split, train=False, stats=None):
        self.df, self.images, self.bscan, self.onset = load_split(split)
        self.train = train
        self.has_labels = "Object_Type" in self.df.columns and self.df["Object_Type"].notna().any()
        if self.has_labels:
            self.y_obj = self.df["Object_Type"].map(OBJ2I).values.astype(np.int64)
            self.y_zone = self.df["Location_Zone"].map(ZONE2I).values.astype(np.int64)
            self.y_illum = self.df["Illumination"].map(ILLUM2I).values.astype(np.int64)
            self.y_dist = self.df["distance"].values.astype(np.float32)
        self.stats = stats

    def __len__(self):
        return len(self.df)

    def _aug_image(self, img):
        if np.random.rand() < 0.8:
            img = np.power(img, np.random.uniform(0.6, 1.6))
        if np.random.rand() < 0.8:
            img = img * np.random.uniform(0.6, 1.5) + np.random.uniform(-0.15, 0.15)
        if np.random.rand() < 0.3:
            img = img + np.random.randn(*img.shape).astype(np.float32) * np.random.uniform(0.01, 0.06)
        img = np.clip(img, 0.0, 1.0)
        if np.random.rand() < 0.25:
            h, w = img.shape[:2]
            eh, ew = np.random.randint(h // 8, h // 3), np.random.randint(w // 8, w // 3)
            y0, x0 = np.random.randint(0, h - eh), np.random.randint(0, w - ew)
            img[y0:y0 + eh, x0:x0 + ew] = np.random.rand()
        return img

    def _aug_bscan(self, bs, on):
        if np.random.rand() < 0.5:
            t = np.random.randint(1, max(2, bs.shape[0] // 8))
            t0 = np.random.randint(0, bs.shape[0] - t)
            bs[t0:t0 + t] = bs.mean(); on[t0:t0 + t] = 1.0
        if np.random.rand() < 0.3:
            bs = bs + np.random.randn(*bs.shape).astype(np.float32) * 0.05
        if np.random.rand() < 0.2:
            keep = np.random.rand(bs.shape[0]) > np.random.uniform(0.1, 0.4)
            bs[~keep] = bs.mean(); on[~keep] = 1.0
        return bs, on

    def __getitem__(self, i):
        img = self.images[i].astype(np.float32) / 255.0
        bs = np.asarray(self.bscan[i], dtype=np.float32)
        on = np.asarray(self.onset[i], dtype=np.float32)
        if self.train:
            img = self._aug_image(img)
            bs, on = self._aug_bscan(bs, on)
        if self.stats is not None:
            bs = (bs - self.stats[0]) / (self.stats[1] + 1e-6)
        item = {
            "image": torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1))),
            "spec": torch.from_numpy(bs).unsqueeze(0),
            "envelope": torch.from_numpy(on).unsqueeze(0),
        }
        if self.has_labels:
            item["object_type"] = torch.tensor(self.y_obj[i])
            item["location_zone"] = torch.tensor(self.y_zone[i])
            item["illumination"] = torch.tensor(self.y_illum[i])
            item["distance"] = torch.tensor(self.y_dist[i])
        return item

def compute_bscan_stats(split="train", n=2000):
    bs = np.load(os.path.join(PROCESSED, f"{split}_bscan.npy"), mmap_mode="r")
    sub = np.asarray(bs[: min(n, len(bs))], dtype=np.float32)
    return float(sub.mean()), float(sub.std())


## 4. สถาปัตยกรรมโมเดล -- ทุกเลเยอร์ด้านล่างเป็น random initialization ทั้งหมด ไม่มี pretrained weight ใดๆ

---

---

**English original below:**

## 4. Model -- everything below is randomly initialised, no pretrained components anywhere

In [ ]:
def conv_bn(cin, cout, k=3, s=1, p=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, p, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True))

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.short = None
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        idt = x if self.short is None else self.short(x)
        o = F.silu(self.b1(self.c1(x)), inplace=True)
        o = self.b2(self.c2(o))
        return F.silu(o + idt, inplace=True)

class ImageEncoder(nn.Module):
    def __init__(self, width=(32, 64, 128, 256), out_dim=256):
        super().__init__()
        w0, w1, w2, w3 = width
        self.stem = nn.Sequential(conv_bn(3, w0, 3, 2, 1), conv_bn(w0, w0, 3, 1, 1))
        self.layer1 = nn.Sequential(BasicBlock(w0, w1, 2), BasicBlock(w1, w1))
        self.layer2 = nn.Sequential(BasicBlock(w1, w2, 2), BasicBlock(w2, w2))
        self.layer3 = nn.Sequential(BasicBlock(w2, w3, 2), BasicBlock(w3, w3))
        self.head = nn.Sequential(nn.Linear(w3 * 2, out_dim), nn.SiLU(inplace=True))
        self.out_dim = out_dim
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        avg = F.adaptive_avg_pool2d(x, 1).flatten(1); mx = F.adaptive_max_pool2d(x, 1).flatten(1)
        return self.head(torch.cat([avg, mx], 1))

class SpecEncoder(nn.Module):
    """2D CNN over the B-scan (ping x range-bin sonar image)."""
    def __init__(self, width=(32, 64, 128), out_dim=192):
        super().__init__()
        w0, w1, w2 = width
        self.stem = conv_bn(1, w0, 3, 1, 1)
        self.layer1 = nn.Sequential(BasicBlock(w0, w0, 2), BasicBlock(w0, w0))
        self.layer2 = nn.Sequential(BasicBlock(w0, w1, 2), BasicBlock(w1, w1))
        self.layer3 = nn.Sequential(BasicBlock(w1, w2, 2), BasicBlock(w2, w2))
        self.head = nn.Sequential(nn.Linear(w2 * 2, out_dim), nn.SiLU(inplace=True))
        self.out_dim = out_dim
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        avg = F.adaptive_avg_pool2d(x, 1).flatten(1); mx = F.adaptive_max_pool2d(x, 1).flatten(1)
        return self.head(torch.cat([avg, mx], 1))

class EnvelopeEncoder(nn.Module):
    """1D CNN over the per-ping first-echo-onset curve."""
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, 9, 2, 4, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 64, 9, 2, 4, bias=False), nn.BatchNorm1d(64), nn.SiLU(inplace=True),
            nn.Conv1d(64, 64, 9, 2, 4, bias=False), nn.BatchNorm1d(64), nn.SiLU(inplace=True),
            nn.Conv1d(64, 128, 9, 2, 4, bias=False), nn.BatchNorm1d(128), nn.SiLU(inplace=True),
            nn.Conv1d(128, 128, 9, 2, 4, bias=False), nn.BatchNorm1d(128), nn.SiLU(inplace=True),
        )
        self.head = nn.Sequential(nn.Linear(128 * 2, out_dim), nn.SiLU(inplace=True))
        self.out_dim = out_dim
    def forward(self, x):
        h = self.net(x)
        avg = F.adaptive_avg_pool1d(h, 1).flatten(1); mx = F.adaptive_max_pool1d(h, 1).flatten(1)
        return self.head(torch.cat([avg, mx], 1))

class EchoVisionNet(nn.Module):
    def __init__(self, n_obj=5, n_zone=4, n_illum=3, dropout=0.2):
        super().__init__()
        self.img = ImageEncoder(); self.spec = SpecEncoder(); self.env = EnvelopeEncoder()
        fdim = self.img.out_dim + self.spec.out_dim + self.env.out_dim
        self.trunk = nn.Sequential(
            nn.Linear(fdim, 512), nn.BatchNorm1d(512), nn.SiLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.SiLU(inplace=True), nn.Dropout(dropout),
        )
        self.head_obj = nn.Linear(256, n_obj)
        self.head_zone = nn.Linear(256, n_zone)
        self.head_illum = nn.Linear(256, n_illum)
        audio_dim = self.spec.out_dim + self.env.out_dim
        self.head_dist = nn.Sequential(nn.Linear(256 + audio_dim, 256), nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(256, 1))

    def forward(self, image, spec, envelope):
        fi = self.img(image); fs = self.spec(spec); fe = self.env(envelope)
        f = torch.cat([fi, fs, fe], 1)
        h = self.trunk(f)
        fa = torch.cat([fs, fe], 1)
        return {
            "object_type": self.head_obj(h),
            "location_zone": self.head_zone(h),
            "illumination": self.head_illum(h),
            "distance": self.head_dist(torch.cat([h, fa], 1)).squeeze(1),
        }

model = EchoVisionNet().to(device)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M  (random init, no pretrained weights)")


## 5. Perception Score (คำนวณตรงตาม metric ของการแข่งขัน)

= 0.4×MacroF1(Object_Type) + 0.3×(1-RMSE(distance)/2) + 0.2×MacroF1(Location_Zone) + 0.1×MacroF1(Illumination)

---

---

**English original below:**

## 5. Perception Score (matches the competition metric exactly)

In [ ]:
def distance_score(y_true, y_pred):
    rmse = float(np.sqrt(np.mean((np.asarray(y_pred) - np.asarray(y_true)) ** 2)))
    return max(0.0, 1.0 - rmse / 2.0), rmse

def perception_score(true, pred):
    f1_obj = f1_score(true["object_type"], pred["object_type"], average="macro")
    f1_zone = f1_score(true["location_zone"], pred["location_zone"], average="macro")
    f1_illum = f1_score(true["illumination"], pred["illumination"], average="macro")
    s_dist, rmse = distance_score(true["distance"], pred["distance"])
    total = 0.40 * f1_obj + 0.30 * s_dist + 0.20 * f1_zone + 0.10 * f1_illum
    return {"perception_score": total, "f1_object_type": f1_obj, "distance_score": s_dist,
            "distance_rmse": rmse, "f1_location_zone": f1_zone, "f1_illumination": f1_illum}


## 6. เทรนโมเดล

เทรนหลาย seed แล้วรวมกัน 2 แบบต่างกัน เพราะ target ทั้งสองประเภทมีพฤติกรรมต่างกัน:

- **Classification** (Object_Type, Location_Zone, Illumination) ดีขึ้นเมื่อเฉลี่ย logits จากหลาย seed (ensemble ลด variance และเพิ่ม Macro F1 ได้จริง)
- **Distance** กลับไม่ได้ประโยชน์จากการ ensemble — ค่า RMSE ของแต่ละ seed แตกต่างกันมาก (0.68-0.84) การเอาโมเดลที่แย่กว่ามาเฉลี่ยรวมกลับทำให้แย่ลง จึงใช้ checkpoint ตัวที่ distance แม่นที่สุดเพียงตัวเดียว

---

---

**English original below:**

## 6. Train

Several models are trained with different random seeds. They are combined two
different ways, because the two target types behave differently:

- **Classification** (`Object_Type`, `Location_Zone`, `Illumination`) improves when
  logits are averaged across seeds -- ensembling reduces variance and lifted macro-F1
  materially.
- **Distance** does *not*. Per-seed distance RMSE varies a lot (0.68 to 0.84 on the
  validation split), and averaging a worse model into a better one drags the result
  down, so the single best-distance checkpoint is used on its own.


In [ ]:
def class_weights(labels, n_cls):
    counts = np.bincount(labels, minlength=n_cls).astype(np.float64)
    counts[counts == 0] = 1.0
    return torch.tensor(counts.sum() / (n_cls * counts), dtype=torch.float32)

@torch.no_grad()
def evaluate(model, loader, dist_mean, dist_std):
    model.eval()
    logits = {"object_type": [], "location_zone": [], "illumination": []}
    preds_dist, trues = [], {"object_type": [], "location_zone": [], "illumination": [], "distance": []}
    for b in loader:
        img = b["image"].to(device); spec = b["spec"].to(device); env = b["envelope"].to(device)
        with torch.autocast(device, dtype=torch.bfloat16):
            out = model(img, spec, env)
        for k in logits:
            logits[k].append(out[k].float().cpu().numpy())
        preds_dist.append(out["distance"].float().cpu().numpy() * dist_std + dist_mean)
        for k in trues:
            trues[k].append(b[k].numpy())
    logits = {k: np.concatenate(v) for k, v in logits.items()}
    preds_dist = np.concatenate(preds_dist)
    trues = {k: np.concatenate(v) for k, v in trues.items()}
    pred = {k: logits[k].argmax(1) for k in logits}
    pred["distance"] = preds_dist
    return perception_score(trues, pred), logits, preds_dist, trues


EPOCHS = 40
BATCH_SIZE = 64
LR = 3e-3
SEEDS = [42, 1, 2, 3]          # each produces one ensemble member

stats = compute_bscan_stats("train")
train_ds = EchoDataset("train", train=True, stats=stats)
val_ds = EchoDataset("val", train=False, stats=stats)
dist_mean = float(train_ds.y_dist.mean()); dist_std = float(train_ds.y_dist.std())
print(f"bscan stats: {stats}  distance mean/std: {dist_mean:.3f}/{dist_std:.3f}")

tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                pin_memory=True, drop_last=True, persistent_workers=True)
vl = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=4,
                pin_memory=True, persistent_workers=True)

w_obj = class_weights(train_ds.y_obj, len(OBJECT_TYPES)).to(device)
w_zone = class_weights(train_ds.y_zone, len(LOCATION_ZONES)).to(device)
w_illum = class_weights(train_ds.y_illum, len(ILLUMINATIONS)).to(device)


def train_one(seed):
    """Train a single member from scratch and return its checkpoint path."""
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = EchoVisionNet().to(device).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR,
                                                total_steps=EPOCHS * len(tl), pct_start=0.25)
    ckpt_path = os.path.join(WEIGHTS_DIR, f"echovision_seed{seed}.pt")
    best = -1.0
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tot = 0.0
        for b in tl:
            img = b["image"].to(device).to(memory_format=torch.channels_last)
            spec = b["spec"].to(device); env = b["envelope"].to(device)
            y_obj = b["object_type"].to(device); y_zone = b["location_zone"].to(device)
            y_illum = b["illumination"].to(device)
            y_dist = (b["distance"].to(device) - dist_mean) / dist_std

            with torch.autocast(device, dtype=torch.bfloat16):
                out = model(img, spec, env)
                loss = (TASK_WEIGHTS["object_type"] * F.cross_entropy(out["object_type"], y_obj, weight=w_obj, label_smoothing=0.05)
                      + TASK_WEIGHTS["location_zone"] * F.cross_entropy(out["location_zone"], y_zone, weight=w_zone, label_smoothing=0.05)
                      + TASK_WEIGHTS["illumination"] * F.cross_entropy(out["illumination"], y_illum, weight=w_illum, label_smoothing=0.05)
                      # plain MSE: the metric is RMSE, so large errors must stay expensive
                      + TASK_WEIGHTS["distance"] * 3.0 * F.mse_loss(out["distance"], y_dist))

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step()
            tot += loss.item()

        res, *_ = evaluate(model, vl, dist_mean, dist_std)
        print(f"  seed{seed} ep{ep:03d} loss={tot/len(tl):.4f} PS={res['perception_score']:.4f} "
              f"obj={res['f1_object_type']:.4f} dist_rmse={res['distance_rmse']:.4f} "
              f"zone={res['f1_location_zone']:.4f} illum={res['f1_illumination']:.4f} {time.time()-t0:.1f}s")
        if res["perception_score"] > best:
            best = res["perception_score"]
            torch.save({"model": model.state_dict(), "stats": stats, "dist_mean": dist_mean,
                        "dist_std": dist_std, "score": best, "seed": seed}, ckpt_path)
    print(f"seed{seed}: best val PS={best:.4f}")
    return ckpt_path


ckpt_paths = [train_one(s) for s in SEEDS]
print("\ntrained members:", ckpt_paths)


## 7. รวมผลจากหลายโมเดล, ปรับ bias ต่อคลาส, แล้วทำนาย

`tune_class_biases` ปรับค่า offset เล็กๆ ต่อคลาสบน validation set เนื่องจาก Macro F1 ให้น้ำหนักทุกคลาสเท่ากัน การขยับ decision boundary เล็กน้อยจึงช่วยเพิ่มคะแนนได้ แม้คลาสต่างๆ จะสมดุลกันเกือบสมบูรณ์อยู่แล้วก็ตาม

---

---

**English original below:**

## 7. Combine members, tune per-class biases, and predict

`tune_class_biases` fits a small additive offset per class on the validation split.
Macro-F1 weights every class equally, so nudging the decision boundary is worth a
few tenths of a point even though the classes are almost perfectly balanced.


In [ ]:
def tune_class_biases(logits, y_true, n_iter=200, grid=np.linspace(-2.0, 2.0, 41), seed=0):
    logits = np.asarray(logits, dtype=np.float64); y_true = np.asarray(y_true)
    n_cls = logits.shape[1]; bias = np.zeros(n_cls)
    def score(b): return f1_score(y_true, (logits + b).argmax(1), average="macro")
    best = score(bias); rng = np.random.default_rng(seed)
    for _ in range(n_iter):
        improved = False
        for c in rng.permutation(n_cls):
            cur = bias[c]
            for g in grid:
                bias[c] = g
                s = score(bias)
                if s > best + 1e-9:
                    best, cur, improved = s, g, True
            bias[c] = cur
        if not improved:
            break
    return bias, best


@torch.no_grad()
def infer(ckpt_path, split):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = EchoVisionNet().to(device)
    model.load_state_dict(ck["model"]); model.eval()
    ds = EchoDataset(split, train=False, stats=ck["stats"])
    dl = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    L = {"object_type": [], "location_zone": [], "illumination": []}
    D = []
    for b in dl:
        img = b["image"].to(device); spec = b["spec"].to(device); env = b["envelope"].to(device)
        with torch.autocast(device, dtype=torch.bfloat16):
            out = model(img, spec, env)
        for k in L:
            L[k].append(out[k].float().cpu().numpy())
        D.append(out["distance"].float().cpu().numpy() * ck["dist_std"] + ck["dist_mean"])
    return {k: np.concatenate(v) for k, v in L.items()}, np.concatenate(D)


HEADS = ["object_type", "location_zone", "illumination"]
vds = EchoDataset("val")
truth = {"object_type": vds.y_obj, "location_zone": vds.y_zone,
         "illumination": vds.y_illum, "distance": vds.y_dist}

val_logits, val_dists = {}, {}
for p in ckpt_paths:
    L, D = infer(p, "val")
    val_logits[p], val_dists[p] = L, D
    print(f"{os.path.basename(p)}: distance RMSE={np.sqrt(np.mean((D - vds.y_dist)**2)):.4f}")

# distance: keep the single best member rather than averaging (averaging was worse)
DIST_CKPT = min(val_dists, key=lambda p: np.sqrt(np.mean((val_dists[p] - vds.y_dist) ** 2)))
print(f"\ndistance model: {os.path.basename(DIST_CKPT)}")

# classification: average logits over all members
avg_val = {h: np.mean([val_logits[p][h] for p in ckpt_paths], 0) for h in HEADS}

biases = {}
for h in HEADS:
    b, s = tune_class_biases(avg_val[h], truth[h])
    biases[h] = b
    print(f"  tuned {h}: macroF1 {s:.4f}")

pred = {h: (avg_val[h] + biases[h]).argmax(1) for h in HEADS}
pred["distance"] = val_dists[DIST_CKPT]
print("\nfinal validation:", perception_score(truth, pred))


In [ ]:
test_logits, test_dists = {}, {}
for p in ckpt_paths:
    L, D = infer(p, "test")
    test_logits[p], test_dists[p] = L, D

avg_test = {h: np.mean([test_logits[p][h] for p in ckpt_paths], 0) for h in HEADS}
obj = (avg_test["object_type"] + biases["object_type"]).argmax(1)
zone = (avg_test["location_zone"] + biases["location_zone"]).argmax(1)
illum = (avg_test["illumination"] + biases["illumination"]).argmax(1)
dist = test_dists[DIST_CKPT]

test_df = pd.read_csv(os.path.join(DATASET, "test.csv"))
sub = pd.DataFrame({
    "image name": test_df["image name"].values,
    "Location_Zone": [LOCATION_ZONES[i] for i in zone],
    "Illumination": [ILLUMINATIONS[i] for i in illum],
    "Object_Type": [OBJECT_TYPES[i] for i in obj],
    "distance": np.round(np.clip(dist, 0.0, None), 2),
})
os.makedirs("outputs", exist_ok=True)
sub.to_csv("outputs/submission.csv", index=False)
print("wrote outputs/submission.csv", sub.shape)
sub.head()


## สรุป: อะไรได้ผล และอะไรไม่ได้ผล (บันทึกไว้เพื่อไม่ต้องทำซ้ำ)

### ✅ สิ่งที่ได้ผลจริง
- **B-scan representation**: จัดรูปสัญญาณเสียง (ping x range-bin) เป็นภาพ 2 มิติ เก็บ timing ที่บอกระยะทางไว้ครบ
- **ใช้ MSE แทน Huber loss** สำหรับ distance เพราะ metric จริงคือ RMSE ที่ลงโทษ error ก้อนใหญ่แบบยกกำลังสอง
- **Multi-seed ensemble สำหรับ classification heads** (Macro F1 เพิ่มขึ้นชัดเจน)
- **Per-class bias tuning** บน validation set
- **แก้ภาพให้รักษาสัดส่วนเดิม (letterbox)** แทนการบีบเป็นจัตุรัส — ภาพต้นฉบับมีสัดส่วนไม่เท่ากันในแต่ละไฟล์ การบีบทำให้ perspective เพี้ยนคนละแบบทุกรูป

### ❌ สิ่งที่ลองแล้วไม่ได้ผล (ลองมาแล้ว 12 วิธีสำหรับ distance)
- ป้อน onset summary statistics เป็น auxiliary feature เข้า distance head
- เพิ่มน้ำหนัก (multiplier) ของ distance loss จาก 3x เป็น 5x หรือ 10x — ยิ่งเพิ่มยิ่งแย่ลง (loss scale ไม่เข้ากับ LR schedule)
- เทรนเพิ่มอีก 6 seed แบบสุ่ม — ไม่มีตัวไหนดีกว่า seed ตั้งต้นเลย
- เทรนนานขึ้น (70 epochs แทน 40)
- ป้อน image features ตรงเข้า distance head
- Gradient Boosting แยกต่างหากบนฟีเจอร์เสียงล้วน (RMSE 0.90 แย่กว่า CNN)
- Calibration แบบต่างๆ (linear / isotonic / class-conditional)
- NNLS stacking รวมทุกโมเดลที่เทรนไว้
- **RasterEncoder อ่าน depth map 2 มิติ** (ping คือการสแกน raster 21×18/24 ของภาพ ไม่ใช่ลำดับเวลา — พิสูจน์แล้วด้วยสถิติ แต่พอเทรนจริงกลับสู้ภาพต้นฉบับไม่ได้)

### ทำไม distance ถึงยากที่สุด
`Clear Path` (ทางเปิดโล่ง ระยะไกล) มีสัดส่วน error ถึง ~49% ทั้งที่มีแค่ ~22% ของข้อมูล เพราะระยะเฉลี่ยไกลถึง 3.8 เมตร ในขณะที่หน้าต่างเวลาฟัง echo แค่ 30 ms ที่ความเร็วเสียง 343 m/s ให้ระยะสูงสุดแค่ ~5.15 เมตร กรณีที่ไกลกว่านี้จึงไม่มี echo ให้อ่านเลย กลายเป็นปัญหาการประมาณความลึกจากภาพล้วนๆ

**ข้อมูลสำคัญที่ยืนยันจาก slide โจทย์:** เสียง ultrasonic ถูก**สังเคราะห์มาจาก depth map ของภาพ** ไม่ใช่วัดจริงจากเซนเซอร์ ดังนั้นภาพจึงเป็น "ต้นทาง" ของข้อมูลระยะทาง ส่วนเสียงเป็นเพียง "สำเนา" ที่สูญเสียความละเอียดไปในกระบวนการสังเคราะห์ — อธิบายได้ว่าทำไมทุกความพยายามปรับปรุงฝั่งเสียงจึงไม่ประสบผลสำเร็จ

## ข้อปฏิบัติตามกฎการแข่งขัน (Compliance notes)

- **ไม่มี pretrained weight**: `EchoVisionNet` และทุก submodule (`ImageEncoder`, `SpecEncoder`, `EnvelopeEncoder`) เป็น `nn.Module` ที่เขียนเองทั้งหมด สร้างจาก `Conv2d`/`Conv1d`/`Linear`/`BatchNorm` แบบ random initialization ไม่มีการใช้ `torchvision.models`, `torch.hub`, หรือโหลด checkpoint จากภายนอกใดๆ ทุก ensemble member เทรนจาก random init ใหม่ทั้งหมด
- **ไม่ใช้ข้อมูลภายนอก**: อ่านเฉพาะ `dataset/images`, `dataset/audio`, และไฟล์ `train.csv` / `val.csv` / `test.csv` ที่โจทย์จัดเตรียมให้เท่านั้น

---

---

**English original below:**

## What worked, and what didn't

Recorded so the result is reproducible and the negative results aren't re-run.

**Worked**
- **B-scan representation.** The 100 kHz recording is a repeating 3000-sample ping
  period; reshaping it into a (ping x range-bin) image keeps echo arrival time as a
  pixel coordinate. A log-mel spectrogram over the raw 11 s waveform would have
  averaged away exactly the timing that encodes range.
- **MSE (not Huber) on distance**, matching the RMSE metric, which punishes large
  errors quadratically.
- **Multi-seed ensembling of the classification heads** (macro-F1 up several points).
- **Per-class bias tuning** on validation.

**Didn't work** (each tried and measured; distance RMSE stayed ~0.68 at best)
- Feeding hand-built onset summary statistics into the distance head.
- Raising the distance loss multiplier from 3x to 5x.
- Four additional random seeds -- all landed at RMSE 0.72-0.84, none beat 0.68.
- Training 70 epochs instead of 40.
- Routing image features directly into the distance head.
- A separate gradient-boosting regressor on hand-built sonar features (RMSE 0.90).
- Isotonic / linear / variance-expanding calibration of the predictions.
- NNLS stacking across every trained model (cross-fitted RMSE 0.69).

**Why distance is the hard target.** `Clear Path` carries ~49% of the total squared
error despite being ~22% of rows: its mean distance is 3.8 m, and at 343 m/s a 30 ms
ping window only reaches ~5.15 m, so the far cases have no echo to read and become a
visual-depth problem instead.

## Compliance notes

- **No pretrained weights**: `EchoVisionNet` and every sub-module (`ImageEncoder`,
  `SpecEncoder`, `EnvelopeEncoder`) are custom `nn.Module`s built from randomly
  initialised `Conv2d`/`Conv1d`/`Linear`/`BatchNorm` layers. No `torchvision.models`,
  no `torch.hub`, no downloaded checkpoints. Every member is trained from a fresh
  random initialisation.
- **No external data**: only `dataset/images`, `dataset/audio`, and the provided
  `train.csv` / `val.csv` / `test.csv` are read.
